# Week 17, Day 3: The common complexity classes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w017d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from math import log2, factorial

n = 20
print(round(log2(n)), n, round(n * log2(n)), n ** 2, n ** 3, 2 ** n, factorial(n))

In [ ]:
def first_and_last(items):
    return items[0], items[-1]            # indexing: O(1)

def price_of(prices, item):
    return prices.get(item, 0)            # dictionary lookup: O(1) on average

def is_even(n):
    return n % 2 == 0                     # arithmetic: O(1)

print(first_and_last([4, 8, 15]), price_of({"tea": 3}, "tea"), is_even(10 ** 50))

In [ ]:
def guesses_needed(secret, low, high):
    count = 0
    while True:
        count += 1
        guess = (low + high) // 2
        if guess == secret:
            return count
        if guess < secret:
            low = guess + 1
        else:
            high = guess - 1

print(guesses_needed(1, 1, 100))
print(guesses_needed(777_777, 1, 1_000_000))
print(max(guesses_needed(s, 1, 1000) for s in range(1, 1001)))

In [ ]:
def stats(items):
    total = 0
    largest = items[0]
    for x in items:                  # one pass
        total += x
        if x > largest:
            largest = x
    return total / len(items), largest

print(stats([3, 9, 4, 6]))

In [ ]:
def has_close_pair(items, gap):
    """Are any two numbers within `gap` of each other?"""
    ordered = sorted(items)                       # O(n log n)
    for i in range(len(ordered) - 1):             # O(n)
        if ordered[i + 1] - ordered[i] <= gap:
            return True
    return False

print(has_close_pair([40, 3, 19, 27, 11], 5), has_close_pair([40, 3, 19, 22], 5))

In [ ]:
def closest_pair_slow(items):
    best = None
    for i in range(len(items)):
        for j in range(i + 1, len(items)):        # every pair
            d = abs(items[i] - items[j])
            if best is None or d < best:
                best = d
    return best

print(closest_pair_slow([40, 3, 19, 27, 11]))

In [ ]:
def zero_triples(items):
    count = 0
    n = len(items)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):              # every triple
                if items[i] + items[j] + items[k] == 0:
                    count += 1
    return count

print(zero_triples([-5, 1, 4, 2, -3, 0, 3]))

In [ ]:
from itertools import permutations

def shortest_tour(dist):
    towns = range(1, len(dist))
    best = None
    for order in permutations(towns):             # (n - 1)! orderings
        route = (0,) + order + (0,)
        length = sum(dist[a][b] for a, b in zip(route, route[1:]))
        if best is None or length < best:
            best = length
    return best

dist = [[0, 2, 9, 10],
        [2, 0, 6, 4],
        [9, 6, 0, 3],
        [10, 4, 3, 0]]
print(shortest_tour(dist))

> **Key idea.** O(1), O(log n), O(n) and O(n log n) **scale**: they handle millions of items. O(n²) and O(n³) handle thousands. O(2ⁿ) and O(n!) handle a few dozen at most.

*Interactive figure on the website: Slide n up and watch when each class stops being practical. The times are rough Python estimates..*

### Exercise: Put them in order

Set `order` to the list of these class names sorted from slowest-growing to fastest-growing: `"O(n^2)"`, `"O(1)"`, `"O(n!)"`, `"O(n log n)"`, `"O(log n)"`, `"O(2^n)"`, `"O(n)"`, `"O(n^3)"`.

In [ ]:
order = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"order\"\n    assert order == [\"O(1)\", \"O(log n)\", \"O(n)\", \"O(n log n)\", \"O(n^2)\", \"O(n^3)\", \"O(2^n)\", \"O(n!)\"]")

### Exercise: Counting digits in O(log n)

Write `digit_count(n)` returning how many digits a whole number of 0 or more has, by repeatedly dividing by 10 with `//` (no `str` or `len`). 0 has 1 digit. Then set `steps_for_a_billion` to how many divisions your loop does for 1,000,000,000.

In [ ]:
steps_for_a_billion = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert digit_count(0) == 1 and digit_count(7) == 1 and digit_count(10) == 2 and digit_count(999) == 3\n    assert digit_count(10 ** 50) == 51\n    assert \"str(\" not in source and \"len(\" not in source\n    assert steps_for_a_billion == 10")

### Exercise: Worst-case guesses

Write `worst_guesses(n)` returning the largest number of guesses `guesses_needed(secret, 1, n)` (from the lesson, included in the starter) needs for any secret from 1 to n. Work it out **without** trying every secret: it's the number of times you can halve, which is `n.bit_length()` (the number of binary digits of n). Check your formula against the brute force for small n.

In [ ]:
def guesses_needed(secret, low, high):
    count = 0
    while True:
        count += 1
        guess = (low + high) // 2
        if guess == secret:
            return count
        if guess < secret:
            low = guess + 1
        else:
            high = guess - 1

def worst_guesses(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"matches brute force\"\n    for n in range(1, 300):\n        assert worst_guesses(n) == max(guesses_needed(s, 1, n) for s in range(1, n + 1)), n\n    assert worst_guesses(1_000_000) == 20 and worst_guesses(10 ** 9) == 30")

### Exercise: How big can n be?

Write `largest_n(kind, budget)` that returns the largest whole number n of 1 or more for which the step count of the given class is at most `budget`, where the step counts are: `"n"`: n; `"n2"`: n²; `"n3"`: n³; `"2n"`: 2ⁿ; `"nlogn"`: n × log₂ n (use `math.log2`). Search upwards by doubling to get past the answer, then halve the gap (binary search) to find it exactly, so it's fast even for a budget of 10¹⁸.

In [ ]:
import math

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert largest_n(\"n\", 10_000_000) == 10_000_000\n    assert largest_n(\"n2\", 10_000_000) == 3162\n    assert largest_n(\"n3\", 10_000_000) == 215\n    assert largest_n(\"2n\", 10_000_000) == 23\n    assert largest_n(\"nlogn\", 10_000_000) == 526_172\n    assert largest_n(\"n2\", 10 ** 18) == 10 ** 9")

### Exercise: Sort first

Write `min_gap(items)` returning the smallest difference between any two numbers in a list of at least two numbers. It must be O(n log n): sort, then one pass comparing neighbours. A list of 200,000 numbers must take well under a second.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert min_gap([40, 3, 19, 27, 11]) == 8\n    assert min_gap([5, 5]) == 0 and min_gap([1, 100]) == 99\n    rng = random.Random(1)\n    data = rng.sample(range(10 ** 9), 200_000)\n    s = sorted(data)\n    assert min_gap(data) == min(b - a for a, b in zip(s, s[1:]))")

### Bug hunt: Exponential inside a loop

This prints the first 35 Fibonacci numbers, but it takes ages: each `fib(i)` is exponential, and it's called 35 times. Make it O(n) overall while keeping the output the same.

In [ ]:
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(35)][-3:])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"same output, without repeating work\"\n    expect_output(\"[2178309, 3524578, 5702887]\")\n    assert \"cache\" in source or \"fib(n - 1) + fib(n - 2)\" not in source, \"fib still recomputes everything\"")

### Bug hunt: Building a string

`stars(n)` builds a line of n stars with `+=` in a loop. In many languages this is O(n²), because each `+` copies the whole string so far. (CPython sometimes avoids the copy, but you can't rely on it.) Rewrite it so it's clearly O(n) without a loop: use string repetition. It must still return the same string.

In [ ]:
def stars(n):
    line = ""
    for i in range(n):
        line += "*"
    return line

print(stars(5))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"same\"\n    expect_output(\"*****\")\n    assert stars(0) == \"\" and len(stars(100_000)) == 100_000\n    assert \"for \" not in source")

### Puzzle: O(n log n) duplicate check

Arrange the lines so `has_duplicate` sorts first, then compares neighbours. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Which class?

Give the class of each, as strings in the style of the first exercise.

In [ ]:
a = ___   # look up a word's definition in a dict
b = ___   # find the shortest round trip through n towns by trying every route
c = ___   # count how many times you can halve n
d = ___   # sorted(items), then one loop over the result
e = ___   # list every subset of n items

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"O(1)\" and b == \"O(n!)\" and c == \"O(log n)\" and d == \"O(n log n)\" and e == \"O(2^n)\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: from cubic to quadratic

`zero_triples` from the lesson is O(n³). Write `count_zero_triples(items)` that gives the same answer for a list of **different** whole numbers in O(n²): sort the list, then for each position `i`, use two positions `lo = i + 1` and `hi` at the end moving towards each other to find pairs that add up to `-items[i]` (like yesterday's challenge). It must handle 1,500 numbers in a few seconds or less, where the O(n³) version would take minutes.

Also write `zero_triples_slow(items)` (the cubic version, for checking) and set `speedup` to how many times fewer triples-or-pairs the fast version looks at for n = 1,500, roughly: n³/6 divided by n²/2, as a whole number.

In [ ]:
speedup = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_small():\n    \"small\"\n    assert count_zero_triples([-5, 1, 4, 2, -3, 0, 3]) == 4\n    assert count_zero_triples([1, 2, 3]) == 0 and count_zero_triples([]) == 0\n    rng = random.Random(3)\n    for _ in range(30):\n        data = rng.sample(range(-40, 41), rng.randint(0, 25))\n        assert count_zero_triples(data) == zero_triples_slow(data)\ndef test_big():\n    \"big\"\n    rng = random.Random(7)\n    data = rng.sample(range(-100_000, 100_001), 1500)\n    assert count_zero_triples(data) == 2094\n    assert speedup == 500")

## Warm-up

For just 20 items, the classes range from 4 steps to more than 2 billion billion. Here they are, slowest-growing first:

**O(1) < O(log n) < O(n) < O(n log n) < O(n²) < O(n³) < O(2ⁿ) < O(n!)**

## O(1) and O(log n)

**Constant, O(1):** the work doesn't depend on n at all. No loop over the data, no hidden loop in a built-in.

(Strictly, arithmetic on enormous whole numbers like 10⁵⁰ costs more than on small ones, since Python stores them digit by digit. For ordinary sizes, treat it as O(1).)

**Logarithmic, O(log n):** each step **cuts the problem down by a fixed fraction**, usually half. log₂ n is "how many times can you halve n before reaching 1": about 10 for a thousand, 20 for a million, 30 for a billion. The classic example is guessing a number between 1 and n by always guessing the middle:

Never more than 10 guesses for a thousand numbers, and at most 20 for a million. This is binary search, the subject of next week's Day 2. Counting the digits of a number by dividing by 10 is also O(log n), with log base 10.

## O(n) and O(n log n)

**Linear, O(n):** a constant amount of work per item: one loop over the data, or a few loops one after another. Most "go through the list and..." tasks are O(n): adding up, finding the largest, counting, filtering, building a dictionary from a list.

You can't do better than O(n) for any task that must look at every item.

**Linearithmic, O(n log n):** typically "split in half log n times, with O(n) work at each level". Merge sort, from last week's divide and conquer, is the example, and so is Python's `sorted`. O(n log n) is only a little more than O(n): for a million items, log₂ n is about 20.

Sorting first is a powerful trick: once the numbers are in order, the closest pair must be **next to each other**, so one pass finds it. Comparing every pair would be O(n²). "Sort first, then one pass" is O(n log n) and solves many problems.

## O(n²) and O(n³)

**Quadratic, O(n²):** usually a loop inside a loop over the same data, or a loop with an O(n) operation inside it (`in` on a list, `insert(0, ...)`, `count`). Comparing every pair, simple sorting methods (next week), and building a string by `+` in a loop all tend to be O(n²).

**Cubic, O(n³):** three nested loops, such as checking every triple, or multiplying two n by n tables of numbers the straightforward way.

Polynomial classes like these are fine for hundreds or a few thousand items, but they don't scale to millions.

## O(2ⁿ) and O(n!)

**Exponential, O(2ⁿ):** usually "try every subset" or a tree recursion where each call makes two more and nothing is remembered. Each extra item **doubles** the work. Plain recursive `fib(n)` is close to this, and so is checking every subset to see if any adds up to a target.

**Factorial, O(n!):** "try every ordering". The classic example is finding the shortest round trip through n towns by trying every route: with 15 towns there are over a trillion orderings.

Exponential and factorial algorithms only work for very small n, roughly 25 and 10 respectively. When a problem seems to need one, look for memoisation (Week 16), a cleverer algorithm, or an answer that's good enough rather than perfect.

## How big can n be?

Python does roughly 10 to 50 million simple steps a second. Using 10 million as a rough, cautious figure, here's the largest n each class can handle in about one second:

| Class | Largest n in about a second | Typical code |
|---|---|---|
| O(1) | any | no loop over the data |
| O(log n) | any | halve each time |
| O(n) | about 10 million | one loop |
| O(n log n) | about 500,000 | sort, then a loop |
| O(n²) | about 3,000 | a loop in a loop |
| O(n³) | about 200 | three nested loops |
| O(2ⁿ) | about 23 | every subset |
| O(n!) | about 10 | every ordering |

This table is how experienced programmers choose an approach: if the input could have a million items, an O(n²) idea is ruled out before writing any code.

## Practice

## Quiz

## Daily challenge

## Before you go

Without writing code, put each of these in a class: finding a name in a phone book by opening it in the middle each time; reading every page of a book; checking every pair of students for a shared birthday; trying every possible 4-digit PIN. Tomorrow: analysing loops in detail, including loops whose inner range changes and loops that don't simply count up by one.